In [21]:
import os
import librosa
import numpy as np
import torch
import soundfile as sf
from torch.nn.functional import interpolate
import torch.nn as nn


In [22]:
# Configurações do Mel-Spectrograma
SAMPLE_RATE = 22050
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512

# Caminho do modelo treinado
MODEL_PATH = 'trained_model.pth'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [23]:
def process_audio_to_mel(audio_path):
    """Carrega o áudio e converte para Mel-Spectrograma normalizado."""
    # Carregar o áudio
    audio, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    
    # Gerar Mel-Spectrograma
    mel_spec = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS
    )
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Normalizar entre 0 e 1
    mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min())
    return mel_spec_db

In [24]:
def reconstruct_audio_from_mel(mel_spec, output_path):
    """Reconstrói o áudio a partir do Mel-Spectrograma e salva no caminho fornecido."""
    mel_spec = librosa.db_to_power(mel_spec)
    audio = librosa.feature.inverse.mel_to_audio(
        mel_spec, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    sf.write(output_path, audio, SAMPLE_RATE)

In [25]:
class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()

        # Encoder
        self.encoder1 = self.conv_block(1, 64)  # Entrada com 1 canal
        self.encoder2 = self.conv_block(64, 128)
        self.encoder3 = self.conv_block(128, 256)

        # Pooling
        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = self.conv_block(256, 512)

        # Decoder
        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.decoder3 = self.conv_block(512, 256)  # Concatena 256 (encoder) + 256 (upconv)

        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.decoder2 = self.conv_block(256, 128)  # Concatena 128 (encoder) + 128 (upconv)

        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.decoder1 = self.conv_block(128, 64)  # Concatena 64 (encoder) + 64 (upconv)

        # Final layer
        self.final_conv = nn.Conv2d(64, 1, kernel_size=1)

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # Encoder
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool(enc1))
        enc3 = self.encoder3(self.pool(enc2))

        # Bottleneck
        bottleneck = self.bottleneck(self.pool(enc3))

        # Decoder
        dec3 = self.upconv3(bottleneck)
        dec3 = torch.cat((dec3, enc3), dim=1)  # Concatena 256 (encoder) + 256 (upconv)
        dec3 = self.decoder3(dec3)

        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)  # Concatena 128 (encoder) + 128 (upconv)
        dec2 = self.decoder2(dec2)

        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)  # Concatena 64 (encoder) + 64 (upconv)
        dec1 = self.decoder1(dec1)

        return self.final_conv(dec1)
    
# Instanciar o modelo
model = UNet().to(DEVICE)


In [26]:
# Carregar o modelo treinado
def load_model():
    """Carrega o modelo treinado."""
    # Certifique-se de que a classe UNet está definida aqui ou importada
    model = UNet().to(DEVICE)
    state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(state_dict)  # Carrega os pesos no modelo
    model.eval()  # Coloca o modelo em modo de avaliação
    return model

In [27]:
# Pipeline Completo
def apply_timbre_transformation(input_audio, output_audio, model):
    """Aplica a transformação de timbre."""
    # Processar o áudio para Mel-Spectrograma
    mel_spec = process_audio_to_mel(input_audio)
    mel_spec_tensor = torch.tensor(mel_spec, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)  # [1, 1, H, W]
    
    # Previsão com o modelo
    with torch.no_grad():
        output_tensor = model(mel_spec_tensor)
    
    # Remover dimensões extras
    output_mel_spec = output_tensor.squeeze(0).squeeze(0).cpu().numpy()
    
    # Reconstruir o áudio e salvar
    reconstruct_audio_from_mel(output_mel_spec, output_audio)

In [28]:
# Caminhos
input_audio_path = '../../data/Hall-Reverb/Hall-Reverb/Bridge/1-0.wav'  # Substitua pelo caminho do áudio de entrada
output_audio_path = 'output_audio.wav'  # Substitua pelo caminho desejado para o áudio transformado

# Carregar o modelo
model = load_model()

# Aplicar a transformação de timbre
apply_timbre_transformation(input_audio_path, output_audio_path, model)
print(f"Áudio transformado salvo em: {output_audio_path}")

C:\Users\Andre\AppData\Local\Temp\ipykernel_2316\520882374.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(MODEL_PATH, map_location=DEVICE)


Áudio transformado salvo em: output_audio.wav
